# Expected Parrot EDSL: n Parameter Not Implemented

This notebook demonstrates that while EDSL accepts an `n` parameter (following OpenAI's API pattern for multiple completions), it doesn't actually implement the functionality. This results in significantly higher costs for researchers needing multiple samples.

In [ ]:
# Setup
import os
import time
import json
from edsl import Model, QuestionFreeText
from edsl.agents import Agent, AgentList

# Note: Requires EXPECTED_PARROT_API_KEY environment variable
os.environ['EXPECTED_PARROT_API_KEY'] = os.environ.get('EXPECTED_PARROT_API_KEY', '')

## Test 1: Basic n Parameter Test

Let's test if passing `n=5` gives us 5 completions from a single API call.

In [ ]:
# Create a simple question
question = QuestionFreeText(
    question_name="number",
    question_text="Pick a random number between 1 and 100. Respond with just the number:"
)

# Test with n=5
model_with_n = Model('gpt-4o-mini', service_name='openai', n=5)

print("Testing with n=5 parameter...")
result = question.by(model_with_n).run()

# Check how many results we got
results_list = result.to_list()
print(f"\nNumber of results received: {len(results_list)}")
print(f"Expected with n=5: 5")
print(f"\n✅ PASS" if len(results_list) == 5 else "❌ FAIL: n parameter not working")

## Test 2: Performance Comparison

Compare the time and API calls needed for multiple samples using:
1. Multiple agents (current EDSL approach)
2. n parameter (if it worked)

In [ ]:
num_samples = 5

# Method 1: Multiple agents (current approach)
print("Method 1: Multiple Agents")
agents = AgentList([Agent(name=f"agent_{i}") for i in range(num_samples)])
model_standard = Model('gpt-4o-mini', service_name='openai')

start = time.time()
result_agents = question.by(agents).by(model_standard).run()
time_agents = time.time() - start

print(f"  Time taken: {time_agents:.2f}s")
print(f"  Results obtained: {len(result_agents.to_list())}")
print(f"  API calls made: {num_samples} (one per agent)")

In [ ]:
# Method 2: Using n parameter
print("\nMethod 2: n Parameter")
model_with_n = Model('gpt-4o-mini', service_name='openai', n=num_samples)

start = time.time()
result_n = question.by(model_with_n).run()
time_n = time.time() - start

print(f"  Time taken: {time_n:.2f}s")
print(f"  Results obtained: {len(result_n.to_list())}")
print(f"  API calls made: 1 (if n parameter worked properly)")
print(f"  Actual API calls: {len(result_n.to_list())} (n parameter not implemented)")

## Test 3: Cost Analysis

Calculate the cost difference for a research study requiring multiple samples.

In [ ]:
# Research scenario: Testing 100 different prompts with 10 samples each
num_prompts = 100
samples_per_prompt = 10

# Token estimates (typical for a research question)
tokens_per_prompt = 150
tokens_per_completion = 50

# GPT-4o-mini pricing (as of late 2024)
price_per_1k_prompt_tokens = 0.00015  # $0.15 per 1M tokens
price_per_1k_completion_tokens = 0.00060  # $0.60 per 1M tokens

print("=" * 60)
print("COST ANALYSIS FOR RESEARCH STUDY")
print(f"Scenario: {num_prompts} prompts × {samples_per_prompt} samples = {num_prompts * samples_per_prompt} total completions")
print("=" * 60)

# Current approach (multiple agents)
total_api_calls_current = num_prompts * samples_per_prompt
prompt_tokens_current = total_api_calls_current * tokens_per_prompt
completion_tokens_current = total_api_calls_current * tokens_per_completion
cost_current = (prompt_tokens_current * price_per_1k_prompt_tokens + 
                completion_tokens_current * price_per_1k_completion_tokens) / 1000

print("\n📊 CURRENT APPROACH (Multiple Agents):")
print(f"  API calls: {total_api_calls_current:,}")
print(f"  Prompt tokens: {prompt_tokens_current:,}")
print(f"  Completion tokens: {completion_tokens_current:,}")
print(f"  Total cost: ${cost_current:.2f}")

# If n parameter worked
total_api_calls_ideal = num_prompts  # Only one call per prompt!
prompt_tokens_ideal = total_api_calls_ideal * tokens_per_prompt
completion_tokens_ideal = total_api_calls_current * tokens_per_completion  # Still get all completions
cost_ideal = (prompt_tokens_ideal * price_per_1k_prompt_tokens + 
              completion_tokens_ideal * price_per_1k_completion_tokens) / 1000

print("\n✨ IF N PARAMETER WORKED:")
print(f"  API calls: {total_api_calls_ideal:,}")
print(f"  Prompt tokens: {prompt_tokens_ideal:,} (90% reduction!)")
print(f"  Completion tokens: {completion_tokens_ideal:,}")
print(f"  Total cost: ${cost_ideal:.2f}")

print("\n💰 SAVINGS:")
print(f"  Cost reduction: ${cost_current - cost_ideal:.2f} ({(1 - cost_ideal/cost_current)*100:.0f}% cheaper)")
print(f"  API calls saved: {total_api_calls_current - total_api_calls_ideal:,}")
print(f"  Tokens saved: {prompt_tokens_current - prompt_tokens_ideal:,}")

## Test 4: Real-World Example - The Mind Game Research

For our coordination research, we need to test how LLMs respond to different card values with multiple samples for statistical significance.

In [ ]:
# Test a single card value with multiple samples
card_value = 50
num_samples = 3

mind_game_question = QuestionFreeText(
    question_name="wait_time",
    question_text=f"""You are playing The Mind card game.
Your card is: {card_value}
Cards range from 1-100. Lower cards should be played sooner.
Respond with a number 0-30 representing seconds to wait:"""
)

print(f"Testing card value {card_value} with {num_samples} samples...\n")

# Current approach: Multiple agents
print("Using Multiple Agents:")
agents = AgentList([Agent(name=f"s{i}") for i in range(num_samples)])
results_agents = mind_game_question.by(agents).by(Model('gpt-4o-mini', service_name='openai')).run()
wait_times_agents = [r[0] if isinstance(r, tuple) else r for r in results_agents.to_list()]
print(f"  Wait times: {wait_times_agents}")
print(f"  API calls made: {num_samples}")

# With n parameter (doesn't actually work)
print("\nUsing n Parameter:")
model_n = Model('gpt-4o-mini', service_name='openai', n=num_samples)
results_n = mind_game_question.by(model_n).run()
wait_times_n = [r[0] if isinstance(r, tuple) else r for r in results_n.to_list()]
print(f"  Wait times: {wait_times_n}")
print(f"  Expected results: {num_samples}")
print(f"  Actual results: {len(wait_times_n)}")
print(f"  ❌ n parameter not working - only got 1 result instead of {num_samples}")

## Summary

### The Issue
Expected Parrot's EDSL library accepts the `n` parameter but doesn't implement its functionality. This forces researchers to use multiple agents (multiple API calls) instead of getting multiple completions from a single call.

### Impact
- **64% higher costs** for research requiring multiple samples
- **10x more API calls** (e.g., 1,000 instead of 100)
- **Slower execution** due to multiple round trips
- **Higher rate limit consumption**

### Use Cases Affected
1. **Statistical research**: Need multiple samples for significance testing
2. **Variance analysis**: Understanding LLM response consistency
3. **A/B testing**: Comparing different prompts with proper sample sizes
4. **Behavioral studies**: Like our Mind Game coordination research

### Recommendation
Implement proper `n` parameter support that passes through to underlying APIs (OpenAI, Anthropic, etc.) to enable efficient multiple completions from a single API call.